# RSOD reproduction on Colab (UATD)

Reproduces the UATD rows of **RSOD** (Li et al., AAAI 2026): labels-only Faster R-CNN vs. RSOD at 1/5/10 % labels.

**Before you start:** `Runtime → Change runtime type → GPU`. Then run the cells from top to bottom.

* Everything that must survive a disconnect lives on Google Drive (`DRIVE_ROOT`): the converted dataset, checkpoints and results.
* If Colab disconnects, **reconnect and run all cells again**: training resumes from the last checkpoint, and finished runs are skipped.
* Steps 1–4 take a few minutes each session. Step 3 (data) is slow only the first time.

In [ ]:
#@title 0. Settings (edit these)
GITHUB_REPO   = "YOUR_GITHUB_USERNAME/sonar-habitat-repro"   #@param {type:"string"}
PCTS          = "5"      #@param {type:"string"}   # e.g. "1 5 10"
SEEDS         = "1"      #@param {type:"string"}   # e.g. "1 2 3"
METHODS       = "sup rsod"  #@param {type:"string"}
ITERS         = ""       #@param {type:"string"}   # "" = paper schedule (180k). e.g. "2000" for a quick test
CKPT_INTERVAL = "2000"   #@param {type:"string"}   # save often so a disconnect loses little
DRIVE_ROOT    = "/content/drive/MyDrive/sonar-habitat"  #@param {type:"string"}
RSOD_COMMIT   = "4e41dfb"  # RSOD commit this kit was written against

import os
os.environ.update(GITHUB_REPO=GITHUB_REPO, PCTS=PCTS, SEEDS=SEEDS, METHODS=METHODS, ITERS=ITERS,
                  CKPT_INTERVAL=CKPT_INTERVAL, DRIVE_ROOT=DRIVE_ROOT, RSOD_COMMIT=RSOD_COMMIT)

In [ ]:
#@title 1. Mount Google Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Get the code (your repo + RSOD)
# Private repo? Add a GitHub token as a Colab secret named GITHUB_TOKEN (key icon on the left).
token = ""
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ""
except Exception:
    pass
url = f"https://{token+'@' if token else ''}github.com/{GITHUB_REPO}.git"
if os.path.isdir('/content/kit/.git'):
    !git -C /content/kit pull -q
else:
    !git clone -q {url} /content/kit
if not os.path.isdir('/content/RSOD/.git'):
    !git clone -q https://github.com/LICZ9/RSOD.git /content/RSOD
!git -C /content/RSOD checkout -q {RSOD_COMMIT} && echo "RSOD at $(git -C /content/RSOD rev-parse --short HEAD)"
!ls /content/kit

In [ ]:
%%bash
# 3. Python 3.10 environment with RSOD's versions (torch 2.0.0 + CUDA 11.8, mmcv 2.1.0, mmengine 0.10.6)
# Colab's own Python is too new for mmcv 2.1.0 wheels, so we make a separate 3.10 environment.
set -e
pip -q install uv
uv python install -q 3.10
[ -x /content/venv310/bin/python ] || uv venv -q /content/venv310 --python 3.10 --seed
P=/content/venv310/bin/python
uv pip install -q --python $P torch==2.0.0 torchvision==0.15.1 --index-url https://download.pytorch.org/whl/cu118
# torch 2.0 imports pkg_resources (removed from recent setuptools) -> pin an older setuptools
uv pip install -q --python $P "setuptools==69.5.1" wheel
uv pip install -q --python $P "numpy==1.26.4" mmengine==0.10.6 cython
uv pip install -q --python $P mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
uv pip install -q --python $P -r /content/RSOD/requirements/runtime.txt
uv pip install -q --python $P --no-build-isolation -e /content/RSOD
$P - <<'EOF'
import torch, mmcv, mmengine, mmdet
from mmcv.ops import RoIAlign, nms  # fails here if mmcv was not built with CUDA ops
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '| mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmdet', mmdet.__version__)
x = torch.rand(10, 4, device='cuda') * 50; x[:, 2:] += x[:, :2]
print('CUDA nms OK:', nms(x, torch.rand(10, device='cuda'), 0.5)[0].shape)
EOF

In [ ]:
#@title 4. Dataset: download UATD once, convert to lossless PNG + COCO on Drive, copy to local disk
import json, os, shutil, subprocess, urllib.request, zipfile
DATA_DRIVE = f"{DRIVE_ROOT}/data/uatd_coco"
if not os.path.exists(f"{DATA_DRIVE}/annotations/instances_test.json"):
    raw = "/content/uatd_raw"; os.makedirs(raw, exist_ok=True)
    files = json.load(urllib.request.urlopen("https://api.figshare.com/v2/articles/21331143/files"))
    for f in files:
        print(f"{f['name']}: {f['size']/1e9:.2f} GB")
    for f in files:
        dst = f"{raw}/{f['name']}"
        if not os.path.exists(dst):
            print('downloading', f['name']); subprocess.run(['wget', '-q', '-O', dst, f['download_url']], check=True)
        if dst.endswith('.zip'):
            with zipfile.ZipFile(dst) as z: z.extractall(raw)
            os.remove(dst)
    !python /content/kit/tools/uatd_to_coco.py --uatd-root {raw} --out {DATA_DRIVE} --to-png
    shutil.rmtree(raw)
splits = [f"{DATA_DRIVE}/annotations/instances_{p}train_seed{s}.json" for p in PCTS.split() for s in SEEDS.split()]
if not all(os.path.exists(s) for s in splits):
    !python /content/kit/tools/make_ssod_splits.py {DATA_DRIVE}/annotations/instances_train.json --percents {PCTS} --seeds {SEEDS}
# Training reads thousands of images; local disk is much faster than Drive.
LOCAL = "/content/data/uatd_coco"
if not os.path.exists(f"{LOCAL}/annotations/instances_test.json"):
    !mkdir -p /content/data && rsync -a {DATA_DRIVE}/ {LOCAL}/
else:
    !rsync -a {DATA_DRIVE}/annotations/ {LOCAL}/annotations/
os.environ['UATD_DATA_ROOT'] = LOCAL
!du -sh {LOCAL} && ls {LOCAL}/annotations | head -20

In [ ]:
#@title 5. Generate the configs inside RSOD
!python /content/kit/tools/gen_rsod_configs.py --rsod-repo /content/RSOD --dataset uatd --coco-train $UATD_DATA_ROOT/annotations/instances_train.json --percents {PCTS} --seeds {SEEDS}

In [ ]:
%%bash
# 6. Train + test (resumes automatically after a disconnect)
# Checkpoints and logs go to Drive. Watch "time:" in the log to see seconds per iteration.
cd /content/RSOD
source /content/venv310/bin/activate
export WORK_ROOT="$DRIVE_ROOT/work_dirs"
bash /content/kit/scripts/run_uatd.sh 2>&1 | grep -v "Warning"

In [ ]:
#@title 7. Progress check (run in a second session or after a stop): last log lines per run
import glob, os
for d in sorted(glob.glob(f"{DRIVE_ROOT}/work_dirs/*")):
    logs = sorted(glob.glob(f"{d}/*/*.log"))
    status = 'DONE' if os.path.exists(f"{d}/DONE") else 'running/incomplete'
    print(f"== {os.path.basename(d)}  [{status}]")
    if logs:
        lines = [l for l in open(logs[-1], errors='ignore') if 'Iter(' in l]
        print('   ', lines[-1].strip()[:220] if lines else '(no iterations logged yet)')

In [ ]:
#@title 8. Results vs. the paper (saved to Drive as results.csv)
!python /content/kit/tools/collect_results.py --work-root {DRIVE_ROOT}/work_dirs --out {DRIVE_ROOT}/results.csv